# Load deps

In [ ]:
# !pip install -q torcheval

In [ ]:
# # # if src modules imported
# # from google.colab import drive
# # drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
import wandb
import torch, os
from types import SimpleNamespace
from diffusers import UNet2DModel
import matplotlib as mpl
import matplotlib.pyplot as plt
from functools import partial
import torchvision.transforms.functional as TF
from torch import nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from datasets import load_dataset

from src.utils import store_attr
from src.datasets import inplace, DataLoaders, show_images, get_grid, show_image
from src.init import clean_mem
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB

# Config

In [ ]:
mpl.rcParams['image.cmap'] = 'gray_r'
dataset_xl,datatset_yl = 'img','label'
dataset_name = "cifar10"
bs = 64
line_sep = "="*50
num_dl_workers = os.cpu_count()
lr = 1e-3
epochs = 1

# Load data

In [ ]:
@inplace
def transformi(b): b[dataset_xl] = [TF.to_tensor(o)-0.5 for o in b[dataset_xl]]

dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)
dt = dls.train
xb,yb = next(iter(dt))
print(f"sample image shape: {xb[0].shape}")
print(line_sep)
show_images(xb[:25]+0.5, figsize=(5,5));

# Training

## Define noisifier

In [ ]:
def linear_sched(betamin=0.0001,betamax=0.02,n_steps=1000):
    beta = torch.linspace(betamin, betamax, n_steps)
    # `SimpleNamespace` allows attribute access (`obj.key`) instead of key access (`obj['key']`)
    return SimpleNamespace(a=1.-beta, abar=(1.-beta).cumprod(dim=0), sig=beta.sqrt())

n_steps = 1000
lin_abar = linear_sched(betamax=0.01)
alphabar = lin_abar.abar
alpha = lin_abar.a
sigma = lin_abar.sig

In [ ]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    return (xt, t.to(device)), ε


(xt,t),ε = noisify(xb[:25],alphabar)
show_images(xt[:25].clip(-0.5, 0.5) + 0.5, imsize=1.5, titles=list(t.cpu().numpy()));

## Init model and dataloaders

In [ ]:
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

def init_ddpm(model):
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            if o.downsamplers:
                for p in list(o.downsamplers): init.orthogonal_(p.conv.weight)
    for o in model.up_blocks:
        for p in o.resnets: p.conv2.weight.data.zero_()
    model.conv_out.weight.data.zero_()
    
def collate_ddpm(b):
    return noisify(default_collate(b)[dataset_xl], alphabar)
def dl_ddpm(ds, nw=num_dl_workers):
    return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=nw)

In [ ]:
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))
# The model we've been using for FashionMNIST
model = UNet(
    in_channels=3, out_channels=3,
    block_out_channels=(32, 64, 128, 256),
    norm_num_groups=8
)
print(f"our model's #parameters: {sum(p.numel() for p in model.parameters())}")
# The default is a much larger model:
default_model = UNet(in_channels=3, out_channels=3)
print(f"defualt model's #parameters: {sum(p.numel() for p in default_model.parameters())}")
clean_mem() # Free up some memory

## Train loop

In [ ]:
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [
    DeviceCB(),
    MixedPrecision(),
    ProgressCB(plot=True),
    MetricsCB(),
    BatchSchedCB(sched)
]
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)
# learn.fit(epochs)

## Sample from the trained model

In [ ]:
@torch.no_grad()
def sample(model, sz):
    ps = next(model.parameters())
    x_t = torch.randn(sz).to(ps)
    preds = []
    for t in reversed(range(n_steps)):
        t_batch = torch.full((x_t.shape[0],), t, device=ps.device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(ps)
        ᾱ_t1 = alphabar[t-1]  if t > 0 else torch.tensor(1)
        b̄_t = 1-alphabar[t]
        b̄_t1 = 1-ᾱ_t1
        noise = model((x_t, t_batch))
        x_0_hat = ((x_t - b̄_t.sqrt() * noise)/alphabar[t].sqrt())
        x_t = x_0_hat * ᾱ_t1.sqrt()*(1-alpha[t])/b̄_t + x_t * alpha[t].sqrt()*b̄_t1/b̄_t + sigma[t]*z
        preds.append(x_t.float().cpu())
    return preds

In [ ]:
# samples = sample(model, (bs, 3, 32, 32))
# s = (samples[-1] + 0.5).clamp(0,1)
# show_images(s[:16], imsize=1.5);

# Add W&B CB to the training pipeline

In [ ]:
class WandBCB(MetricsCB):
    order=100
    def __init__(self, config, *ms, project='ddpm_cifar10', **metrics):
        store_attr()
        super().__init__(*ms, **metrics)
        
    def before_fit(self, learn):
        super().before_fit(learn)
        wandb.init(project=self.project, config=self.config)
    def after_fit(self, learn): wandb.finish()

    def _log(self, d): wandb.log(d)
    
    def after_epoch(self, learn):
        prefix = 'train_' if learn.model.training else 'eval_'
        log = {
            prefix+k:f'{v.compute():.3f}'
            for k,v in self.all_metrics.items()
        }
        log['epoch'] = learn.epoch
        self._log(log)
        if not learn.model.training:
            self._log({'samples':self.sample_figure(learn)})

        
    def sample_figure(self, learn):
        with torch.no_grad():
            samples = sample(learn.model, (16, 3, 32, 32))
        s = (samples[-1] + 0.5).clamp(0,1)
        plt.clf()
        fig, axs = get_grid(16)
        for im,ax in zip(s[:16], axs.flat): show_image(im, ax=ax)
        return fig

    def after_batch(self, learn):
        super().after_batch(learn) 
        wandb.log({'loss':learn.loss})

In [ ]:
epochs = 2
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
wandbcb =  WandBCB(config={'lr':lr, 'epochs':epochs, 'comments':'default unet logging test'})
cbs = [DeviceCB(), MixedPrecision(), ProgressCB(plot=True), wandbcb, BatchSchedCB(sched)]
model = UNet(
    in_channels=3, out_channels=3,
    block_out_channels=(32, 64, 128, 256),
    norm_num_groups=8
)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)
learn.fit(epochs)